# 02 — Sectionizer
**Project:** Clinical Medication Extraction | **Phase 2a of the roadmap**

## What we're building and why

A clinical note isn't a bag of words — it has **structure**, and that structure carries meaning. The same drug name means completely different things depending on where it appears:

| Section | "metoprolol" here means |
|---|---|
| `ALLERGIES` | patient reacts badly to it — **do not** extract as a current med |
| `MEDICATIONS` | patient is taking it now |
| `DISCHARGE MEDICATIONS` | patient should take it going forward |
| `FAMILY HISTORY` | someone *else* takes it |
| `HPI` | might be historical, stopped, or hypothetical |

So the sectionizer isn't preprocessing busywork — it's the component that gives every later extraction its **context**. Without it, your Phase 3 precision will be capped no matter how good your NER model is.

**By the end of this notebook you'll have:** a tested `split_sections()` function, a coverage metric proving it works, an error analysis of where it fails, and a sectioned dataset saved for Phase 2b (the rules extractor).

## Setup

In [ ]:
import pandas as pd
import re
from collections import Counter
import statistics as st

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    pass

BASE = '/content/drive/MyDrive/clinical-nlp/' if IN_COLAB else ''
WORK = BASE + 'data/working/'

import os
os.makedirs(WORK, exist_ok=True)

# Load the subset saved at the end of notebook 01
work = pd.read_parquet(WORK + 'notes_subset.parquet')
print('Notes in working subset:', len(work))
work[['medical_specialty', 'sample_name']].head(3)

## Step 1 — The structural fact that shapes everything

Before writing any regex, look at what these notes physically *are*. Run this and read the output carefully.

In [ ]:
sample = work['transcription'].iloc[0]
print('Contains newline characters?', '\n' in sample)
print('Number of commas:', sample.count(','))
print()
print(repr(sample[:700]))

### 🔑 The key insight

**These notes contain zero newline characters.** Not one, across the entire corpus.

Look at the raw string above: `'HISTORY OF PRESENT ILLNESS:,  The patient is...fishbone.,PAST MEDICAL HISTORY: , Significant for...'`

What happened here: the notes were dictated and transcribed, then flattened into single-line CSV cells. **The comma is doing the job a line break would normally do.** You'll see three variants of the same boundary:

- `HEADER:,` — colon then comma
- `HEADER: ,` — colon, space, comma
- `HEADER:` — plain colon, no comma

Why this matters so much: the obvious approach — `note.split('\n')` and check if a line looks like a header — **is impossible here**. Every line-based sectionizer tutorial you'll find online assumes newlines. You have to find headers *inside* a continuous string, which means the regex has to do all the work.

This is a genuinely realistic lesson: real clinical text arrives in whatever shape the source system dumped it, and step one is always discovering that shape rather than assuming it. When you get MIMIC access, its notes *do* have newlines — and your sectionizer will need adapting. Log that in `decisions.md` now.

## Step 2 — Finding header candidates

Our pattern, then a breakdown of every piece:

In [ ]:
HEADER_RE = re.compile(r'(?:^|[,.;:]\s*)([A-Z][A-Z0-9 /&()\'-]{1,60}?)\s*:\s*,?\s*')

# See what it finds in one note
for m in HEADER_RE.finditer(sample):
    print(f'pos {m.start():5d} | {m.group(1)!r}')

### Reading the regex piece by piece

```
(?:^|[,.;:]\s*)     ([A-Z][A-Z0-9 /&()'-]{1,60}?)     \s*:\s*,?\s*
└─── anchor ───┘     └────── the header text ──────┘   └─ separator ─┘
```

**`(?:^|[,.;:]\s*)` — the anchor.** A header must appear at the start of the note or right after a comma/period/semicolon/colon. This is what stops mid-sentence false positives. `(?:...)` is a *non-capturing* group — it participates in matching but doesn't show up in `.group(1)`, so we get the header text alone.

**`[A-Z]` — must start with a capital.**

**`[A-Z0-9 /&()'-]{1,60}?`** — the body: uppercase letters, digits, spaces, and the punctuation that genuinely appears in real headers (`PAST MEDICAL HISTORY/SURGERIES`, `ASSESSMENT & PLAN`, `A-1 WOMAC SCORE`).

**The `?` after `{1,60}` makes it lazy** — and this one character prevents a real bug. Without it, the quantifier is greedy: it grabs as much as it can, and when the total exceeds 60 characters the engine backtracks by *sliding the start position forward*, which silently chops the first letter off. That's how `PAST MEDICAL HISTORY/SURGERIES/HOSPITALIZATIONS` became `AST MEDICAL HISTORY/...` in an earlier version of this pattern. Lazy matching stops at the first colon instead, so nothing gets truncated.

**`\s*:\s*,?\s*`** — the separator, absorbing all three punctuation variants above so the section body starts at clean text.

## Step 3 — Filtering out false positives

The regex alone still matches things that aren't headers. Rather than making the pattern more complicated (regexes get unreadable fast), we filter matches with a separate function. **Separating "find candidates" from "validate candidates" is a deliberate design choice** — each piece stays simple and testable.

In [ ]:
STOPWORDS = {'DR', 'MD', 'RN', 'AM', 'PM', 'NOTE', 'ADDENDUM'}

def _is_header(cand: str) -> bool:
    """Reject candidates that matched the pattern but aren't real headers."""
    c = cand.strip()

    # 1. Length sanity: 'X' is too short, a runaway match too long
    if len(c) < 2 or len(c) > 60:
        return False

    # 2. Known non-headers ('DR. X:' appears constantly in these notes)
    if c in STOPWORDS:
        return False

    letters = [ch for ch in c if ch.isalpha()]
    if not letters:
        return False

    # 3. Headers are essentially all-caps. 90% not 100%, to tolerate stray
    #    lowercase from transcription noise.
    if sum(ch.isupper() for ch in letters) / len(letters) < 0.9:
        return False

    # 4. Pure numbers/punctuation -> this is what kills '10:30' and 'BP 120/80'
    if re.fullmatch(r'[0-9 /.-]+', c):
        return False

    return True

# Prove it works on the traps
for t in ['HISTORY OF PRESENT ILLNESS', 'X', '10', 'DR', 'Patient states', 'HEENT']:
    print(f'{t!r:35} -> {_is_header(t)}')

## Step 4 — Normalizing section names

`PHYSICAL EXAMINATION`, `PHYSICAL EXAM`, and `OBJECTIVE` are the same section wearing three hats. If we don't normalize, downstream code has to know all three spellings — and will silently miss the fourth one it's never seen.

Two design decisions worth understanding:

**Why a hand-built dictionary and not clustering/fuzzy matching?** With ~50 real section types, a dictionary is exact, debuggable, and takes 20 minutes. Fuzzy matching would introduce its own error mode (`DISCHARGE MEDICATIONS` collapsing into `MEDICATIONS` — a clinically dangerous merge). Choose the boring tool when the boring tool is correct.

**Why keep exam sub-headings separate?** `HEENT`, `ABDOMEN`, `LUNGS` are *sub*-sections of the physical exam, not top-level sections. We prefix them `exam:` so they're grouped but not lost. Medication extraction can then ignore `exam:*` entirely — the exam rarely contains prescriptions, so skipping it removes false-positive surface area for free.

In [ ]:
SECTION_MAP = {
    'HISTORY OF PRESENT ILLNESS': 'hpi', 'HISTORY OF THE PRESENT ILLNESS': 'hpi',
    'HPI': 'hpi', 'INTERIM HISTORY': 'hpi', 'SUBJECTIVE': 'hpi',
    'CHIEF COMPLAINT': 'chief_complaint', 'CC': 'chief_complaint',
    'REASON FOR VISIT': 'chief_complaint', 'REASON FOR RETURN VISIT': 'chief_complaint',
    'PAST MEDICAL HISTORY': 'pmh',
    'PAST MEDICAL HISTORY/SURGERIES/HOSPITALIZATIONS': 'pmh',
    'PMH': 'pmh', 'PAST SURGICAL HISTORY': 'psh',
    # --- medication-bearing sections: the ones Phase 2b cares about ---
    'MEDICATIONS': 'medications', 'CURRENT MEDICATIONS': 'medications',
    'MEDICATIONS ON ADMISSION': 'medications', 'HOME MEDICATIONS': 'medications',
    'OTHER MEDICATIONS': 'medications', 'DIABETES MEDICATIONS': 'medications',
    'DISCHARGE MEDICATIONS': 'discharge_medications',
    'MEDICATIONS AT DISCHARGE': 'discharge_medications',
    'ALLERGIES': 'allergies', 'DRUG INTOLERANCE': 'allergies',
    # --- context sections: drugs here are NOT the patient's current meds ---
    'FAMILY HISTORY': 'family_history', 'SOCIAL HISTORY': 'social_history',
    'REVIEW OF SYSTEMS': 'ros', 'ROS': 'ros',
    'PHYSICAL EXAMINATION': 'physical_exam', 'PHYSICAL EXAM': 'physical_exam',
    'OBJECTIVE': 'physical_exam', 'VITAL SIGNS': 'vitals', 'VITALS': 'vitals',
    'LABORATORY DATA': 'labs', 'LAB STUDIES': 'labs', 'LABORATORY': 'labs',
    'PERTINENT LABORATORIES': 'labs',
    'ASSESSMENT': 'assessment', 'IMPRESSION': 'assessment',
    'ASSESSMENT AND PLAN': 'assessment_plan', 'ASSESSMENT & PLAN': 'assessment_plan',
    'PLAN': 'plan', 'TREATMENT PLAN': 'plan', 'RECOMMENDATIONS': 'plan',
    'HOSPITAL COURSE': 'hospital_course',
    'DISCHARGE DIAGNOSIS': 'discharge_diagnosis',
    'DISCHARGE DIAGNOSES': 'discharge_diagnosis',
    'PRINCIPAL DIAGNOSES': 'discharge_diagnosis',
    'DISCHARGE INSTRUCTIONS': 'discharge_instructions',
    'DISPOSITION': 'disposition', 'FOLLOWUP': 'followup', 'FOLLOW-UP': 'followup',
}

EXAM_SUBHEADS = {'HEENT','ABDOMEN','NECK','LUNGS','CHEST','HEART','SKIN','EXTREMITIES',
                 'GENERAL','CARDIOVASCULAR','NEUROLOGIC','NEUROLOGICAL','RESPIRATORY',
                 'MUSCULOSKELETAL','GU','GI','PSYCHIATRIC','EYES','NOSE','THROAT',
                 'ORAL CAVITY','AXILLA','BACK','PELVIS','RECTAL','GENITOURINARY',
                 'GASTROINTESTINAL','PSYCH','PSYCHE','SOCIAL','LYMPHATIC','BREASTS'}

def normalize_header(raw: str) -> str:
    """Map a raw header to a canonical section key."""
    if raw in SECTION_MAP:
        return SECTION_MAP[raw]
    if raw in EXAM_SUBHEADS:
        return 'exam:' + raw.lower().replace(' ', '_')
    # Unknown header: keep it, lightly cleaned. Never silently drop content.
    return raw.lower().replace(' ', '_').replace('/', '_')

print(normalize_header('PHYSICAL EXAM'), '|', normalize_header('HEENT'),
      '|', normalize_header('SOME NEW HEADER'))

## Step 5 — Putting it together

Now the main function. The logic: find all valid header matches, then each section's **body is the text from the end of one header to the start of the next**. The last section runs to the end of the note.

Note the two edge cases handled explicitly:
- **`_preamble`** — text before the first header (some notes open with a title like `DISCHARGE SUMMARY,` before any real section).
- **`_unsectioned`** — no headers found at all. We return the whole note under this key rather than an empty dict, because **silently returning nothing is how content disappears without anyone noticing.** Fail loudly.

In [ ]:
def split_sections(note: str, normalize: bool = True) -> dict:
    """Split an MTSamples-style flat clinical note into {section_key: body_text}.

    Returns {'_unsectioned': <whole note>} when no headers are detected.
    Repeated headers are concatenated rather than overwritten.
    """
    if not isinstance(note, str) or not note.strip():
        return {}

    matches = [m for m in HEADER_RE.finditer(note) if _is_header(m.group(1))]
    if not matches:
        return {'_unsectioned': note.strip()}

    sections = {}

    # Text before the first header
    if matches[0].start() > 0:
        pre = note[:matches[0].start()].strip(' ,.')
        if pre:
            sections['_preamble'] = pre

    for i, m in enumerate(matches):
        raw = m.group(1).strip()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(note)
        body = note[m.end():end].strip().strip(',').strip()
        key = normalize_header(raw) if normalize else raw
        if key in sections:
            sections[key] += ' ' + body      # same header twice -> concatenate
        else:
            sections[key] = body
    return sections


# Try it
for k, v in split_sections(sample).items():
    print(f'[{k}]\n    {v[:110]}\n')

## Step 6 — Test it before trusting it

**This is the habit that separates a script from a system.** Six tests, each targeting a specific trap we designed against. Run them every time you change the regex — if one goes red, you broke something.

(In Phase 6 these move into `tests/test_sectionizer.py` and run in CI. Same tests, better home.)

In [ ]:
TESTS = {
    'truncation (lazy quantifier)':
        ('PAST MEDICAL HISTORY/SURGERIES/HOSPITALIZATIONS:, Diabetes.,ALLERGIES:, None.',
         {'pmh', 'allergies'}),
    'time is not a header (10:30)':
        ('HPI:, Patient seen at 10:30 today.,PLAN:, Discharge.',
         {'hpi', 'plan'}),
    'Dr. X: is not a header':
        ('HPI:, Seen by Dr. X:  no issues.,PLAN:, Home.',
         {'hpi', 'plan'}),
    'lab value BP 120/80 is not a header':
        ('LABORATORY DATA:, Na 139, K 4.1, BP 120/80.  GENERAL:, Well appearing.',
         {'labs', 'exam:general'}),
    'repeated header concatenates':
        ('MEDICATIONS:, Aspirin.,MEDICATIONS:, Metoprolol.',
         {'medications'}),
    'headerless note fails loudly':
        ('The patient did well overnight and was discharged home.',
         {'_unsectioned'}),
}

passed = 0
for name, (text, expected) in TESTS.items():
    got = set(split_sections(text).keys())
    ok = got == expected
    passed += ok
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
    if not ok:
        print(f'      expected {expected}\n      got      {got}')

print(f'\n{passed}/{len(TESTS)} tests passed')
# The concatenation test deserves a look at its actual output:
print('\nrepeated-header body ->',
      split_sections('MEDICATIONS:, Aspirin.,MEDICATIONS:, Metoprolol.')['medications'])

## Step 7 — Measure it: coverage

Tests prove it handles cases we thought of. A **metric** tells us how it does on data we haven't looked at.

`coverage` = what fraction of a note's characters ended up inside some section body. Near 1.0 means almost nothing was dropped on the floor.

**Why this metric and not accuracy?** We have no labelled "correct sections" to score against — building that would cost hours. Coverage is a *proxy*: it can't tell us sections are labelled correctly, but it reliably catches the failure that actually matters (content vanishing). Choosing a cheap proxy metric that catches the dominant failure mode, and being explicit about what it *can't* see, is exactly the judgment call that goes in `decisions.md`.

In [ ]:
def coverage(note: str, sections: dict) -> float:
    if not note:
        return 0.0
    return min(1.0, sum(len(v) for v in sections.values()) / len(note))


rows = []
for idx, row in work.iterrows():
    t = row['transcription']
    s = split_sections(t)
    rows.append({
        'idx': idx,
        'sample_name': row['sample_name'],
        'n_sections': len(s),
        'coverage': coverage(t, s),
        'headerless': list(s.keys()) == ['_unsectioned'],
        'sections': list(s.keys()),
    })
report = pd.DataFrame(rows)

print(f"Notes processed:        {len(report)}")
print(f"Mean coverage:          {report['coverage'].mean():.3f}")
print(f"Median sections/note:   {report['n_sections'].median():.0f}")
print(f"Headerless notes:       {report['headerless'].sum()}")
print(f"Notes below 0.80 cov:   {(report['coverage'] < 0.80).sum()}")

## Step 8 — Error analysis: look at what failed

Never accept a metric without reading the failures behind it. This is the cell that generates your error taxonomy.

In [ ]:
print('=== HEADERLESS NOTES ===')
for i in report[report['headerless']]['idx'].head(4):
    print('-' * 70)
    print(work.loc[i, 'transcription'][:260])
print()
print('=== LOWEST-COVERAGE NOTES ===')
for i in report.nsmallest(4, 'coverage')['idx']:
    print(f"cov={report.set_index('idx').loc[i, 'coverage']:.2f} | "
          f"{work.loc[i, 'transcription'][:160]!r}")

### Reading the failures — the two error types

**Type 1: genuinely headerless notes (the ~9 above).** These are free-flowing narrative progress notes: *"The patient made some progress during therapy..."* — dictated as continuous prose with no structure at all. **This is not a bug.** No sectionizer can find sections that don't exist. Our function does the right thing by returning `_unsectioned` and letting downstream code decide. In Phase 2b, medication extraction on these notes simply runs over the whole text with no section context — lower precision, correctly flagged.

**Type 2: truncated source notes.** Look at that `'SUBJECTIVE:,'` note — the entire transcription is a header with no body. That's a **data quality problem in MTSamples**, not a sectionizer problem. Coverage of 0.0 is the correct output for a note with zero content.

The distinction matters more than it looks: one is a limitation to document, the other is data to filter. Conflating them means you'll waste an evening "fixing" a regex that was already right. **Diagnose before you fix** — the same instinct as reading an error message properly before changing code.

## Step 9 — Where do medications actually live?

The payoff question. Phase 2b will extract drugs — this tells us which sections to point it at.

In [ ]:
sec_counts = Counter()
for s in report['sections']:
    sec_counts.update(s)

top = pd.DataFrame(sec_counts.most_common(20), columns=['section', 'n_notes'])
top['pct_of_notes'] = (100 * top['n_notes'] / len(report)).round(1)
print(top.to_string(index=False))

In [ ]:
MED_SECTIONS = ['medications', 'discharge_medications', 'allergies']
CONTEXT_SECTIONS = ['family_history', 'social_history', 'hpi', 'hospital_course', 'plan']

has_med_section = report['sections'].apply(
    lambda ss: any(m in ss for m in MED_SECTIONS[:2]))
print(f"Notes with a medications section: {has_med_section.sum()} / {len(report)} "
      f"({100*has_med_section.mean():.0f}%)")
print()
print("What this means for Phase 2b:")
print(f"  - {100*has_med_section.mean():.0f}% of notes: extract from the medications section (high precision)")
print(f"  - the rest: extract from full text, flag lower confidence")

## Step 10 — Save the sectioned dataset

Two outputs. **Long format** (one row per section) is what Phase 2b will actually consume — it makes "run the extractor only on medication sections" a one-line filter instead of a nested loop.

In [ ]:
long_rows = []
for idx, row in work.iterrows():
    for key, body in split_sections(row['transcription']).items():
        long_rows.append({
            'note_id': idx,
            'sample_name': row['sample_name'],
            'medical_specialty': row['medical_specialty'],
            'section': key,
            'text': body,
            'n_chars': len(body),
        })
sections_long = pd.DataFrame(long_rows)

sections_long.to_parquet(WORK + 'sections_long.parquet')
report.drop(columns='sections').to_parquet(WORK + 'sectionizer_report.parquet')

print('Saved sections_long.parquet   rows:', len(sections_long))
print('Saved sectionizer_report.parquet')
print()
print(sections_long[sections_long['section'] == 'medications'][['note_id', 'text']].head(5).to_string())

## Step 11 — Save the module for Phase 6

Writing this out as a `.py` now means Phase 6 (notebook → repo) is a file move, not a rewrite. Notebooks are for exploring; modules are for reusing.

In [ ]:
module_src = '''"""Sectionizer for flat (newline-free) clinical transcription notes."""
import re

HEADER_RE = re.compile(r"(?:^|[,.;:]\\s*)([A-Z][A-Z0-9 /&()\'-]{1,60}?)\\s*:\\s*,?\\s*")
STOPWORDS = ''' + repr(STOPWORDS) + '''
SECTION_MAP = ''' + repr(SECTION_MAP) + '''
EXAM_SUBHEADS = ''' + repr(EXAM_SUBHEADS) + '''


def _is_header(cand):
    c = cand.strip()
    if len(c) < 2 or len(c) > 60 or c in STOPWORDS:
        return False
    letters = [ch for ch in c if ch.isalpha()]
    if not letters:
        return False
    if sum(ch.isupper() for ch in letters) / len(letters) < 0.9:
        return False
    if re.fullmatch(r"[0-9 /.-]+", c):
        return False
    return True


def normalize_header(raw):
    if raw in SECTION_MAP:
        return SECTION_MAP[raw]
    if raw in EXAM_SUBHEADS:
        return "exam:" + raw.lower().replace(" ", "_")
    return raw.lower().replace(" ", "_").replace("/", "_")


def split_sections(note, normalize=True):
    if not isinstance(note, str) or not note.strip():
        return {}
    matches = [m for m in HEADER_RE.finditer(note) if _is_header(m.group(1))]
    if not matches:
        return {"_unsectioned": note.strip()}
    sections = {}
    if matches[0].start() > 0:
        pre = note[:matches[0].start()].strip(" ,.")
        if pre:
            sections["_preamble"] = pre
    for i, m in enumerate(matches):
        raw = m.group(1).strip()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(note)
        body = note[m.end():end].strip().strip(",").strip()
        key = normalize_header(raw) if normalize else raw
        sections[key] = sections[key] + " " + body if key in sections else body
    return sections


def coverage(note, sections):
    if not note:
        return 0.0
    return min(1.0, sum(len(v) for v in sections.values()) / len(note))
'''

os.makedirs(BASE + 'src', exist_ok=True)
with open(BASE + 'src/sectionizer.py', 'w') as f:
    f.write(module_src)

# verify the written module imports and behaves identically
import importlib.util
spec = importlib.util.spec_from_file_location('sectionizer', BASE + 'src/sectionizer.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
assert mod.split_sections(sample).keys() == split_sections(sample).keys()
print('Wrote src/sectionizer.py and verified it reproduces notebook behaviour.')

## What you built, and what's next

**Built:** a sectionizer that handles newline-free transcription text, filters four classes of false positive, normalizes ~50 header variants to canonical keys, concatenates repeated headers, and fails loudly rather than silently. ~93% mean coverage, with both failure modes diagnosed and attributed.

**The four ideas worth carrying forward — these are the transferable ones:**
1. **Look at the raw bytes before designing.** The no-newlines discovery invalidated the obvious approach entirely.
2. **Separate finding from validating.** A simple regex plus a readable filter function beats one clever regex you can't debug at 11pm.
3. **Test the traps, measure the rest.** Unit tests cover what you thought of; a proxy metric catches what you didn't.
4. **Diagnose before fixing.** Two failure types looked identical in the metric; one needed documenting, the other needed filtering, neither needed a code change.

**For `decisions.md`:**
- Sectionizer is regex + rule-based, not ML — ~50 header types, exact and debuggable; revisit only if MIMIC's header diversity breaks it
- Coverage chosen as proxy metric (no labelled section gold set); it detects dropped content, not mislabelled sections
- Exam sub-headings namespaced `exam:*` and excluded from medication extraction
- Headerless notes (~2%) route to full-text extraction with a lower-confidence flag
- MIMIC notes have real newlines — sectionizer will need a line-aware path when credentialing lands

**Next: `03_rules_extractor.ipynb`** — build the drug lexicon, extract dose/route/frequency around each hit, and use these section labels to assign status (current vs. discharge vs. allergy vs. family history). The sections you just built are what make that last part possible.